# 03 — Chunking and Document Loading

## Why this notebook exists

Notebook 02 ranked whole documents by embedding each one as a single vector. That worked because our documents are tiny and each is about one thing. Real documents aren't: a support page covers warranty *and* hours *and* spare-part depots, and a PDF manual runs for pages. Squash all of that into one vector and it blurs — a question about spare parts and a question about warranty both match the same averaged document, and specific details get washed out.

The fix is to break documents into smaller, focused **chunks** and embed those, so retrieval can return the exact passage that answers a question. But first we have to *get the documents in* — including a PDF, which needs real extraction. This notebook does both: load the corpus (markdown + a PDF, via `pypdf`) into text with `source`/`page` metadata, split it into well-sized overlapping chunks, and show chunk-level retrieval returning the precise passage — with a citation — instead of a whole averaged document.

Requires an `OPENAI_API_KEY` (we embed chunks at the end to show the payoff). Still no vector database — just NumPy.

## What you'll learn

- How to **load** a mixed corpus — markdown files and a **PDF** (with `pypdf`) — into a uniform list of text units carrying `source` and `page` metadata.
- Why a single embedding per document is too coarse, and why **chunking** improves retrieval granularity.
- Three **chunking strategies**: naive fixed-size character splits, **token-based** splits (`tiktoken`), and **recursive/structure-aware** splits (`langchain-text-splitters`) — and the role of **chunk overlap**.
- How to attach **metadata** (`source`, `page`, `chunk_id`) to every chunk so retrieved passages can be cited.
- How **chunk-level retrieval** returns the exact relevant passage — the fix for notebook 02's whole-document blur.

## 1. Setup

We reuse notebook 02's embedding toolkit — `embed`, `embed_many`, and `cosine` — re-declared inline so this notebook stands alone. We load `OPENAI_API_KEY` from the environment (and from a local `.env` if `python-dotenv` is installed). Document loading and chunking themselves need no API; the key is used only at the end (Section 6) to embed chunks and show the retrieval payoff.

In [ ]:
import os

# Optional: load a local .env if python-dotenv is installed. Real env vars win.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# ── Key guard ──────────────────────────────────────────────────────────────
if not os.environ.get("OPENAI_API_KEY"):
    print("=" * 60)
    print("OPENAI_API_KEY is not set.")
    print("=" * 60)
    print()
    print("This notebook embeds chunks at the end (Section 6) to show the")
    print("retrieval payoff. Loading and chunking themselves need no key.")
    print()
    print("Set it and restart the kernel:")
    print("  export OPENAI_API_KEY=sk-...")
    raise SystemExit("Set OPENAI_API_KEY and restart the kernel to continue.")

print("OPENAI_API_KEY set ✓")

In [ ]:
import numpy as np
from openai import OpenAI

EMBED_MODEL = "text-embedding-3-small"

openai_client = OpenAI()  # reads OPENAI_API_KEY from the environment


def embed(text: str) -> np.ndarray:
    """Embed a single string into a NumPy vector."""
    resp = openai_client.embeddings.create(model=EMBED_MODEL, input=text)
    return np.array(resp.data[0].embedding, dtype=np.float32)


def embed_many(texts: list) -> np.ndarray:
    """Embed a list of strings in ONE API call; returns a (len(texts), dim) matrix."""
    resp = openai_client.embeddings.create(model=EMBED_MODEL, input=texts)
    return np.array([d.embedding for d in resp.data], dtype=np.float32)


def cosine(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity between two vectors (higher = more similar)."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


print("Setup OK")
print(f"Embedding model: {EMBED_MODEL}")